In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load Welsh CEFR dataset from HuggingFace
ds = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]
df_main = ds.to_pandas()  

# Load your B2 JSON data
df_b2 = pd.read_json("b2_welsh.json")
df_b2["cefr_level"] = "B2"
df_b2 = df_b2.drop_duplicates(subset="text", keep="first")

# Combine both DataFrames
df_combined = pd.concat([df_main, df_b2], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset="text", keep="first")

# Convert back to HuggingFace Dataset
ds_merged = Dataset.from_pandas(df_combined)

In [3]:
ds_merged

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 2020
})

In [4]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in ds_merged["cefr_level"]])

In [5]:
model_name = "./eurobert_cefr_spanish_only/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [6]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [7]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [8]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [9]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(ds_merged, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = ds_merged.select(train_idx)
    ds_val = ds_merged.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_spanish_welsh_b2/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in ["A1", "A2", "B1", "B2", "C1", "C2"]:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 404/404 [00:00<00:00, 4598.52 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50168\1177059815.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.403200,1.216306,0.480198,0.430087,0.550225,0.480198,0.483986,0.888889,0.626728,0.358696,0.272727,0.309859,0.000000,0.000000,0.000000,0.806452,0.192308,0.310559,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.902800,0.831855,0.601485,0.594830,0.598439,0.601485,0.643617,0.790850,0.709677,0.431193,0.388430,0.408696,0.000000,0.000000,0.000000,0.700935,0.576923,0.632911,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.656200,0.767258,0.680693,0.679244,0.693809,0.680693,0.704918,0.843137,0.767857,0.531746,0.553719,0.542510,0.000000,0.000000,0.000000,0.831579,0.607692,0.702222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 2...


Map: 100%|██████████| 404/404 [00:00<00:00, 10159.84 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50168\1177059815.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Map: 100%|██████████| 404/404 [00:00<00:00, 9072.12 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50168\1177059815.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.378200,0.870681,0.589109,0.559040,0.564871,0.589109,0.626374,0.750000,0.682635,0.456140,0.214876,0.292135,0.000000,0.000000,0.000000,0.593939,0.748092,0.662162,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.922900,0.838873,0.658416,0.662313,0.677991,0.658416,0.693252,0.743421,0.717460,0.507042,0.595041,0.547529,0.000000,0.000000,0.000000,0.818182,0.618321,0.704348,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.723800,0.740210,0.690594,0.683057,0.681253,0.690594,0.738372,0.835526,0.783951,0.571429,0.462810,0.511416,0.000000,0.000000,0.000000,0.716418,0.732824,0.724528,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 404/404 [00:00<00:00, 11541.81 examples/s]
C:\Users\c24082331\AppData\Local\Temp\ipykernel_50168\1177059815.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.446600,1.012836,0.522277,0.428339,0.457526,0.522277,0.479100,0.980263,0.643629,0.166667,0.008264,0.015748,0.000000,0.000000,0.000000,0.701149,0.465649,0.559633,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.916300,0.852552,0.618812,0.575128,0.587247,0.618812,0.628019,0.855263,0.724234,0.466667,0.173554,0.253012,0.000000,0.000000,0.000000,0.651316,0.755725,0.699647,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.734400,0.817331,0.658416,0.652212,0.649815,0.658416,0.707602,0.796053,0.749226,0.514563,0.438017,0.473214,0.000000,0.000000,0.000000,0.707692,0.702290,0.704981,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [10]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_spanish_welsh_b2/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [11]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)

In [12]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.693809  0.680693  0.679244  0.704918  0.843137  0.767857   
1        2        0.641683  0.655941  0.638315  0.685000  0.895425  0.776204   
2        3        0.654073  0.655941  0.647897  0.654639  0.835526  0.734104   
3        4        0.681253  0.690594  0.683057  0.738372  0.835526  0.783951   
4        5        0.649815  0.658416  0.652212  0.707602  0.796053  0.749226   
5  Average        0.664127  0.668317  0.660145  0.698106  0.841133  0.762268   

         A2                      ...   B1        B2                      \
  Precision    Recall        F1  ...   F1 Precision    Recall        F1   
0  0.531746  0.553719  0.542510  ...  0.0  0.831579  0.607692  0.702222   
1  0.536585  0.363636  0.433498  ...  0.0  0.688525  0.646154  0.666667   
2  0.515152  0.418033  0.461538  ...  0.0  0.783784  0.669231  0.721992   
3  0.571429  0.462810  0.511416  ...  0.0  0.716418  0.732824  0.724528   
4  0.514563  0.438017  0.473214  ...  0.0  0.707692  0.702290  0.704981   
5  0.533895  0.447243  0.484435  ...  0.0  0.745600  0.671638  0.704078   

         C1                    C2              
  Precision Recall   F1 Precision Recall   F1  
0       0.0    0.0  0.0       0.0    0.0  0.0  
1       0.0    0.0  0.0       0.0    0.0  0.0  
2       0.0    0.0  0.0       0.0    0.0  0.0  
3       0.0    0.0  0.0       0.0    0.0  0.0  
4       0.0    0.0  0.0       0.0    0.0  0.0  
5       0.0    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]